<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026/%D0%A7%D0%B0%D1%81%D1%82%D1%8C_III_%D0%98%D0%BD%D1%81%D1%82%D1%80%D1%83%D0%BC%D0%B5%D0%BD%D1%82%D0%B0%D1%80%D0%B8%D0%B9_%D0%BE%D1%86%D0%B5%D0%BD%D0%BA%D0%B8_%D0%B8_%D0%B2%D1%8B%D0%B1%D0%BE%D1%80%D0%B0_%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ниже — объединённый, доработанный и готовый к публикации текст первой части главы «Кросс‑валидация, bias‑variance tradeoff и выбор гиперпараметров». Он вбирает лучшие стороны обоих обсуждённых вариантов: строгость, прозрачность выкладок, наглядные аналогии и подробные «Углублённые взгляды». Можно копировать и вставлять в книгу как есть.

---

# Глава 5. Кросс‑валидация, bias‑variance tradeoff и выбор гиперпараметров  
## Часть I. Bias‑variance tradeoff: теоретический фундамент

В предыдущих главах мы строили модели — от линейной регрессии до SVM — и оценивали их качество на тестовой выборке. Но почему одна модель работает лучше другой? Почему добавление признаков иногда улучшает, а иногда ухудшает предсказания? Ответ кроется в фундаментальном разложении ошибки на смещение, дисперсию и неустранимый шум. Понимание этого компромисса — ключ к осознанному выбору сложности модели и стратегии валидации.

### 1. Постановка задачи и ожидаемая ошибка

Пусть истинная зависимость между вектором признаков $\mathbf{x} \in \mathbb{R}^D$ и целевой переменной $y$ имеет вид
$$
y = f(\mathbf{x}) + \varepsilon,
$$
где $f(\mathbf{x})$ — неизвестная детерминированная функция, а $\varepsilon$ — случайный шум, не зависящий от $\mathbf{x}$, с нулевым математическим ожиданием и постоянной дисперсией:
$$
\mathbb{E}[\varepsilon] = 0, \qquad \mathrm{Var}(\varepsilon) = \sigma^2.
$$
Шум моделирует все неучтённые факторы, погрешности измерений и внутреннюю стохастичность явления. Его дисперсия $\sigma^2$ задаёт нижнюю границу достижимой точности любого алгоритма.

Обучающая выборка $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^n$ порождается независимыми реализациями случайной пары $(\mathbf{x}, y)$ из некоторого распределения $P(\mathbf{x}, y)$. По ней строится модель $\hat{f}(\mathbf{x}; \mathcal{D})$, которая для любого нового объекта $\mathbf{x}_0$ даёт предсказание $\hat{y}_0 = \hat{f}(\mathbf{x}_0; \mathcal{D})$. Качество предсказания естественно измерять квадратичной функцией потерь $(y_0 - \hat{f}(\mathbf{x}_0; \mathcal{D}))^2$. Поскольку и обучающая выборка $\mathcal{D}$, и тестовый шум $\varepsilon_0$ случайны, истинной мерой точности модели служит **ожидаемая ошибка предсказания** (expected prediction error) в точке $\mathbf{x}_0$:
$$
\text{Err}(\mathbf{x}_0) = \mathbb{E}_{\mathcal{D},\,\varepsilon_0}\!\Bigl[\bigl(y_0 - \hat{f}(\mathbf{x}_0; \mathcal{D})\bigr)^2\Bigr],
\tag{1.1}
$$
где математическое ожидание берётся по всем возможным обучающим выборкам фиксированного объёма $n$ и по всем реализациям шума в тестовой точке. Далее для краткости будем писать $\mathbb{E}$ вместо $\mathbb{E}_{\mathcal{D},\varepsilon_0}$ и $\hat{f}(\mathbf{x}_0)$ вместо $\hat{f}(\mathbf{x}_0; \mathcal{D})$.

Именно эту величину мы хотели бы минимизировать, но напрямую вычислить её невозможно — доступна лишь одна реализация обучающей выборки. Разложение ожидаемой ошибки, которое мы сейчас получим, позволит понять, из каких составляющих она складывается и как на них влиять.

### 2. Разложение ожидаемой квадратичной ошибки

Покажем, что $\text{Err}(\mathbf{x}_0)$ допускает аддитивное разложение на три принципиально разных слагаемых. Введём обозначение для среднего предсказания модели по всем обучающим выборкам:
$$
\bar{f}(\mathbf{x}_0) = \mathbb{E}_{\mathcal{D}}[\hat{f}(\mathbf{x}_0)].
$$
Начнём с тождественного преобразования разности между истинным значением и предсказанием:
$$
\begin{aligned}
y_0 - \hat{f}(\mathbf{x}_0) &= \bigl(y_0 - f(\mathbf{x}_0)\bigr) + \bigl(f(\mathbf{x}_0) - \bar{f}(\mathbf{x}_0)\bigr) + \bigl(\bar{f}(\mathbf{x}_0) - \hat{f}(\mathbf{x}_0)\bigr) \\
&= \varepsilon_0 + \text{Bias}(\mathbf{x}_0) + \text{Variance‑term}(\mathbf{x}_0),
\end{aligned}
$$
где смещение $\text{Bias}(\mathbf{x}_0) = f(\mathbf{x}_0) - \bar{f}(\mathbf{x}_0)$ и отклонение конкретной модели от её среднего $\bar{f}(\mathbf{x}_0) - \hat{f}(\mathbf{x}_0)$. Возведём сумму в квадрат и возьмём математическое ожидание. При раскрытии скобок перекрёстные члены исчезнут благодаря следующим фактам:

- $\mathbb{E}_{\varepsilon_0}[\varepsilon_0] = 0$, и шум $\varepsilon_0$ не зависит от $\mathcal{D}$, поэтому $\mathbb{E}[\varepsilon_0 \cdot g(\mathcal{D})] = 0$ для любой функции $g$;
- $\mathbb{E}_{\mathcal{D}}[\bar{f}(\mathbf{x}_0) - \hat{f}(\mathbf{x}_0)] = 0$ по определению $\bar{f}$.

Таким образом,
$$
\begin{aligned}
\text{Err}(\mathbf{x}_0) &= \mathbb{E}\bigl[\varepsilon_0^2\bigr] + \bigl(f(\mathbf{x}_0) - \bar{f}(\mathbf{x}_0)\bigr)^2 + \mathbb{E}\bigl[(\bar{f}(\mathbf{x}_0) - \hat{f}(\mathbf{x}_0))^2\bigr] \\
&= \underbrace{\sigma^2}_{\text{Шум}} \;+\; \underbrace{\bigl(\mathbb{E}[\hat{f}(\mathbf{x}_0)] - f(\mathbf{x}_0)\bigr)^2}_{\text{Квадрат смещения (bias}^2\text{)}} \;+\; \underbrace{\mathbb{E}\bigl[(\hat{f}(\mathbf{x}_0) - \mathbb{E}[\hat{f}(\mathbf{x}_0)])^2\bigr]}_{\text{Дисперсия (variance)}}.
\end{aligned}
\tag{2.1}
$$

Три слагаемых имеют ясный содержательный смысл.

- **Неустранимый шум** $\sigma^2$ определяет минимально возможную ошибку: даже зная истинную функцию $f$, мы не можем предсказать случайную компоненту $\varepsilon$. Никакая модель не способна опуститься ниже этого порога.
- **Квадрат смещения** показывает, насколько среднее предсказание модели систематически отклоняется от истинной функции. Высокое смещение означает, что модель в принципе не способна адекватно описать целевую зависимость — это характеризует **недообучение**.
- **Дисперсия** измеряет, насколько сильно предсказания модели меняются при обучении на разных выборках одного и того же объёма. Высокая дисперсия говорит о чрезмерной чувствительности к конкретному набору данных — это характеризует **переобучение**.

Соотношение (2.1) — центральный результат теории bias‑variance tradeoff для квадратичной функции потерь. Оно показывает, что уменьшение одного из компонентов почти неизбежно ведёт к росту другого, и задача состоит в нахождении оптимального баланса.

> **Углублённый взгляд: полное доказательство разложения**  
> Проведём вывод формально, без апелляции к интуитивным соображениям. Пусть $y_0 = f(\mathbf{x}_0) + \varepsilon_0$, где $\varepsilon_0$ не зависит от $\mathcal{D}$, $\mathbb{E}[\varepsilon_0]=0$, $\mathrm{Var}(\varepsilon_0)=\sigma^2$. Обозначим $\hat{f} = \hat{f}(\mathbf{x}_0; \mathcal{D})$ и $\bar{f} = \mathbb{E}_{\mathcal{D}}[\hat{f}]$. Тогда
> $$
> \begin{aligned}
> \text{Err}(\mathbf{x}_0) &= \mathbb{E}\bigl[(y_0 - \hat{f})^2\bigr] = \mathbb{E}\bigl[(f + \varepsilon_0 - \hat{f})^2\bigr] \\
> &= \mathbb{E}\bigl[(\varepsilon_0 + (f - \hat{f}))^2\bigr] \\
> &= \mathbb{E}[\varepsilon_0^2] + 2\mathbb{E}[\varepsilon_0 (f - \hat{f})] + \mathbb{E}[(f - \hat{f})^2].
> \end{aligned}
> $$
> Поскольку $\varepsilon_0$ не зависит от $\mathcal{D}$ и, следовательно, от $\hat{f}$, второе слагаемое равно $2\,\mathbb{E}[\varepsilon_0]\mathbb{E}[f - \hat{f}] = 0$. Первое слагаемое даёт $\sigma^2$. Остаётся $\mathbb{E}[(f - \hat{f})^2]$. Добавим и вычтем $\bar{f}$:
> $$
> \begin{aligned}
> \mathbb{E}[(f - \hat{f})^2] &= \mathbb{E}\bigl[(f - \bar{f} + \bar{f} - \hat{f})^2\bigr] \\
> &= (f - \bar{f})^2 + 2(f - \bar{f})\underbrace{\mathbb{E}[\bar{f} - \hat{f}]}_{=0} + \mathbb{E}[(\bar{f} - \hat{f})^2] \\
> &= \bigl(\mathbb{E}[\hat{f}] - f\bigr)^2 + \mathbb{E}\bigl[(\hat{f} - \mathbb{E}[\hat{f}])^2\bigr].
> \end{aligned}
> $$
> Здесь мы воспользовались тем, что $f$ и $\bar{f}$ — детерминированные величины для фиксированного $\mathbf{x}_0$, а $\mathbb{E}[\hat{f}] = \bar{f}$ по определению. В результате получено в точности разложение (2.1).

### 3. Интуиция и геометрическая аналогия

Чтобы лучше понять смысл смещения и дисперсии, часто проводят аналогию со стрельбой по мишени. Представьте, что каждая обучающая выборка — это один «выстрел» (обученная модель), а центр мишени — истинная функция $f(\mathbf{x}_0)$.

- **Смещение** характеризует, насколько далеко в среднем ложатся попадания от центра мишени. Если прицел сбит, попадания образуют кучную группу, но в стороне от цели.
- **Дисперсия** показывает разброс попаданий. Даже при идеально настроенном прицеле дрожание рук даст широкий разброс.

Возможны четыре сочетания, представленные в таблице.

| Смещение | Дисперсия | Интерпретация |
|----------|-----------|---------------|
| Низкое | Низкая | Идеальная модель: точна и устойчива |
| Низкое | Высокая | Модель в среднем близка к истине, но сильно зависит от выборки (переобучение) |
| Высокое | Низкая | Модель стабильна, но систематически ошибается (недообучение) |
| Высокое | Высокая | Модель одновременно неточна и нестабильна — худший случай |

Сложность модели непосредственно управляет этим балансом. Простая модель — например, линейная регрессия с одним признаком — обладает ограниченной гибкостью и не может точно приблизить сложную функцию $f$: её смещение велико. Но предсказания мало меняются при варьировании обучающей выборки — дисперсия низка. Напротив, очень сложная модель — глубокое решающее дерево или полином степени 20 — способна почти идеально подогнаться под обучающие точки, что даёт низкое смещение. Однако малейшие изменения в данных приводят к совсем другой аппроксимации, то есть дисперсия высока.

С ростом сложности смещение, как правило, монотонно убывает, а дисперсия монотонно растёт. Ожидаемая ошибка, будучи их суммой (плюс $\sigma^2$), имеет U‑образный профиль. Оптимальная сложность соответствует минимуму суммы и достигается там, где снижение смещения уже не компенсирует рост дисперсии. Этот феномен и называют **bias‑variance tradeoff**.

### 4. Bias‑variance для классификации

В задачах классификации чаще используется индикаторная функция потерь (0‑1 loss): $L(y, \hat{y}) = \mathbf{1}\{y \neq \hat{y}\}$. Для неё аддитивное разложение вида (2.1) невозможно, поскольку потери принимают лишь значения 0 и 1, а математическое ожидание квадрата ошибки не совпадает с вероятностью ошибки. Тем не менее, интуитивные представления о смещении и дисперсии сохраняют силу.

Рассмотрим метод $k$ ближайших соседей. При $k = 1$ классификатор подстраивается под каждый обучающий пример, граница сильно изрезана: смещение близко к нулю (решение локально точно), но дисперсия велика — небольшое изменение выборки может кардинально изменить границу. При большом $k$ граница сглаживается, смещение растёт (решение перестаёт улавливать тонкие детали распределения), зато дисперсия падает. Таким образом, компромисс проявляется и здесь.

Для вероятностных классификаторов, выдающих оценку вероятности $\hat{p}(y|\mathbf{x})$, можно применить разложение для log‑loss или Brier score, которые являются выпуклыми и тесно связаны с MSE. В частности, ожидаемый log‑loss также раскладывается на неустранимую энтропию, величину, аналогичную смещению, и дисперсию предсказанных вероятностей.

> **Углублённый взгляд: bias‑variance разложение для 0‑1 loss (Domingos, 2000)**  
> Домингос предложил разложение ожидаемой 0‑1 ошибки, основанное на понятии «главного предсказания» — наиболее частого ответа модели при фиксированном $\mathbf{x}_0$ по всем возможным обучающим выборкам. Ошибка распадается на несколько слагаемых:
> $$
> \text{Err}_{0/1} = \sigma^2 + \text{Bias}^2 + \text{Variance} + \text{Covariance},
> $$
> где $\sigma^2$ — неустранимый шум (вероятность того, что истинная метка не совпадает с оптимальной байесовской), Bias$^2$ — систематическое отклонение главного предсказания от байесовского оптимума, Variance — изменчивость предсказаний вокруг главного, а Covariance возникает из‑за неаддитивности 0‑1 потерь. Особенность этого разложения в том, что дисперсия может действовать как в положительную, так и в отрицательную сторону: иногда более изменчивая модель даёт меньшую ошибку, чем стабильная. Это объясняет случаи, когда увеличение сложности модели не приводит к росту ошибки даже при нулевом смещении. На практике классическое интуитивное представление о bias‑variance tradeoff остаётся полезным, но требует осторожности в деталях.

### 5. Практические следствия

Понимание bias‑variance разложения даёт непосредственные инструменты для диагностики и улучшения моделей.

- **Диагностика типа ошибки.** Если модель демонстрирует высокую ошибку как на обучении, так и на тесте — это признак высокого смещения (недообучение). Напротив, если ошибка на обучении очень мала, а на тесте резко возрастает — доминирует дисперсия (переобучение).
- **Регуляризация.** Штраф за сложность ($L_2$, $L_1$, early stopping) целенаправленно уменьшает дисперсию, жертвуя смещением. Параметр регуляризации $\lambda$ напрямую управляет положением на кривой bias‑variance tradeoff: при $\lambda = 0$ модель максимально сложна (низкое смещение, высокая дисперсия), с ростом $\lambda$ сложность падает, смещение растёт, дисперсия убывает. Оптимальное $\lambda$ уравновешивает эти два эффекта.
- **Объём выборки.** Дисперсия убывает с ростом $n$ (закон больших чисел), тогда как смещение от $n$ не зависит — оно определяется исключительно классом моделей. Поэтому на больших выборках можно позволить себе более сложные модели: рост дисперсии будет скомпенсирован объёмом данных, а выигрыш в смещении приведёт к лучшему итоговому качеству. Именно поэтому глубокие нейронные сети требуют огромных датасетов.

Таким образом, bias‑variance разложение — это не просто теоретическая конструкция, а рабочий инструмент для анализа поведения моделей, выбора их сложности и понимания влияния регуляризации и размера данных.

---

## Вопросы для самопроверки

### Для начинающих
1. Что измеряет смещение (bias) модели и что — дисперсия (variance)? Какой тип ошибки соответствует каждому из них?
2. Опишите, как выглядит переобучение и недообучение с точки зрения bias‑variance разложения.
3. Почему добавление регуляризации обычно уменьшает дисперсию, но может увеличить смещение?
4. Какая компонента ожидаемой ошибки не зависит ни от выбора модели, ни от объёма обучающей выборки?
5. Приведите пример модели с высоким смещением и низкой дисперсией, и пример модели с низким смещением и высокой дисперсией.

### Для профессионалов
1. Выведите полностью разложение ожидаемой среднеквадратичной ошибки (2.1) из определений, обосновывая каждый переход.
2. Покажите, что в разложении MSE перекрёстный член $\mathbb{E}[\varepsilon_0 \cdot (f - \hat{f})]$ действительно равен нулю, используя независимость шума и обучающей выборки.
3. Объясните, почему для 0‑1 потерь аддитивное разложение вида bias²+variance не работает, и опишите основные идеи разложения Домингоса (2000).
4. Объясните, используя закон больших чисел, почему с ростом объёма выборки дисперсия предсказаний убывает, а смещение остаётся неизменным. Как этот факт влияет на выбор модели при больших $n$?

Ниже представлена вторая часть главы, продолжающая тему после разложения bias‑variance. Она содержит необходимые теоретические сведения, математические выкладки, «Углублённый взгляд» и практический код с демонстрациями.

---

## Часть II. Кросс‑валидация для независимых данных: k‑fold, LOOCV и стратификация

Понимание компромисса между смещением и дисперсией, изложенное в предыдущей части, подводит нас к практической задаче: как, имея лишь одну конечную выборку, оценить ожидаемую ошибку модели и выбрать её оптимальную сложность? Однократное разбиение на обучающую и тестовую выборки даёт оценку, слишком сильно зависящую от случайности разбиения. Кросс‑валидация предлагает систематическое решение, позволяя полнее использовать данные и получать более устойчивые оценки качества.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import (KFold, StratifiedKFold, LeaveOneOut,
                                     RepeatedKFold, cross_val_score)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
```

### 1. Зачем нужна кросс‑валидация?

Простейший способ оценить качество модели — случайно разделить данные на обучающую и тестовую выборки, обучить модель на первой и измерить ошибку на второй. Такой подход страдает от двух существенных недостатков.

- **Высокая дисперсия оценки.** Ошибка, вычисленная по одному конкретному разбиению, может заметно отличаться от ошибки при другом разбиении. Мы получаем не надёжную характеристику модели, а лишь одно случайное значение, которое может ввести в заблуждение при сравнении алгоритмов.
- **Неэффективное использование данных.** Часть наблюдений не участвует в обучении, что особенно критично при малых объёмах выборки. Модель, обученная на меньшем числе примеров, как правило, хуже, и её оценка оказывается смещённой.

Идея кросс‑валидации состоит в том, чтобы повторить процедуру обучения и тестирования несколько раз на разных подмножествах одних и тех же данных и усреднить полученные значения ошибки. Это снижает дисперсию оценки и позволяет использовать все наблюдения как для обучения, так и для проверки. В результате мы получаем более точную и стабильную оценку ожидаемой ошибки, что крайне важно для выбора модели и настройки гиперпараметров.

### 2. k‑fold кросс‑валидация

Наиболее распространённый вариант — **k‑fold кросс‑валидация**. Исходная выборка случайным образом разбивается на $k$ непересекающихся блоков (фолдов) примерно равного размера. Для каждого $i = 1, \dots, k$ выполняется:

- фолд $i$ объявляется тестовым;
- остальные $k-1$ фолдов объединяются в обучающую выборку;
- модель обучается на этой обучающей выборке, и вычисляется ошибка $\text{Err}_i$ на тестовом фолде.

Итоговая оценка кросс‑валидации есть среднее арифметическое:
$$
\text{CV}_{(k)} = \frac{1}{k} \sum_{i=1}^{k} \text{Err}_i.
\tag{2.1}
$$
Предельный случай $k = n$, когда каждый тестовый фолд состоит ровно из одного наблюдения, называется **leave‑one‑out кросс‑валидацией (LOOCV)**. При $k = 5$ или $k = 10$ говорят о 5‑fold и 10‑fold кросс‑валидации соответственно.

**Свойства k‑fold CV.**

- **Смещение оценки.** При малом $k$ обучающая выборка в каждом испытании имеет размер $n(1 - 1/k)$, что меньше полного объёма данных. Поскольку уменьшение обучающей выборки обычно ухудшает модель, оценка ошибки оказывается пессимистичной (завышенной) относительно ожидаемой ошибки модели, обученной на всей выборке. С ростом $k$ размер обучающих подвыборок приближается к $n$, и смещение уменьшается. LOOCV обладает наименьшим смещением среди k‑fold методов.
- **Дисперсия оценки.** При большом $k$ обучающие выборки в разных фолдах сильно перекрываются, что приводит к высокой корреляции между ошибками $\text{Err}_i$. В результате дисперсия среднего значения $\text{CV}_{(k)}$ может быть значительной. Эмпирически и теоретически показано, что 5‑ или 10‑fold CV часто имеют меньшую дисперсию, чем LOOCV, несмотря на большее число слагаемых.

- **Стандартная ошибка CV.** Для оценки неопределённости часто вычисляют стандартную ошибку среднего:
  $$
  \text{SE} = \frac{\text{std}(\text{Err}_1, \dots, \text{Err}_k)}{\sqrt{k}}.
  $$
  Однако из‑за корреляции ошибок на разных фолдах эта формула даёт заниженное значение стандартной ошибки и не должна интерпретироваться как доверительный интервал в классическом смысле. Её можно использовать как грубый индикатор стабильности оценки, но для формального сравнения моделей требуются более аккуратные методы.

> **Углублённый взгляд: дисперсия LOOCV для линейной регрессии**
>
> Покажем, что дисперсия оценки LOOCV может быть выше, чем у k‑fold. Рассмотрим линейную регрессию с вектором параметров $\hat{\boldsymbol{\beta}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$ и матрицей проекции $\mathbf{H} = \mathbf{X}(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T$. Известно, что LOOCV‑ошибка для одного наблюдения $i$ может быть записана через соответствующее диагональное значение $h_{ii}$ матрицы $\mathbf{H}$:
> $$
> \text{Err}_i^{\text{LOO}} = \frac{y_i - \hat{y}_i}{1 - h_{ii}},
> $$
> где $\hat{y}_i$ — предсказание, полученное с использованием всех $n$ наблюдений. Тогда
> $$
> \text{CV}_{\text{LOO}} = \frac{1}{n} \sum_{i=1}^n \left( \frac{y_i - \hat{y}_i}{1 - h_{ii}} \right)^2.
> $$
> Дисперсия этой величины зависит от распределения $h_{ii}$ и остатков. Если в данных присутствуют точки с большим значением $h_{ii}$ (высокий leverage), знаменатель $1 - h_{ii}$ становится малым, и соответствующие слагаемые начинают доминировать, вызывая резкий рост дисперсии. В k‑fold CV каждая точка участвует в тесте ровно один раз, но в обучающей выборке отсутствует целый блок, что делает отдельные оценки менее зависимыми друг от друга. Следовательно, 5‑ или 10‑fold CV часто дают оценку с меньшей дисперсией, чем LOOCV, несмотря на меньшее число фолдов.

### 3. Стратифицированная кросс‑валидация

При решении задач классификации с несбалансированными классами случайное разбиение на фолды может привести к тому, что в каком‑либо тестовом (или обучающем) фолде окажется крайне мало представителей миноритарного класса или не окажется вовсе. Оценка ошибки в таком фолде становится неинформативной, а модель — плохо обученной.

**Стратифицированная кросс‑валидация** гарантирует, что доля каждого класса сохраняется примерно одинаковой во всех фолдах. Технически это достигается разбиением отдельно внутри каждого класса, после чего части объединяются в итоговые фолды с сохранением общей пропорции. Такой подход особенно важен при малых $k$ и сильном дисбалансе. В `sklearn` для этого предназначен класс `StratifiedKFold`.

### 4. Повторяющаяся кросс‑валидация (Repeated k‑fold)

Оценка, полученная однократной k‑fold кросс‑валидацией, всё ещё может варьироваться в зависимости от конкретного случайного разбиения. Для дальнейшего снижения дисперсии процедуру повторяют $r$ раз с разными начальными случайными разбиениями, после чего результаты усредняют:
$$
\text{CV}_{(k, r)} = \frac{1}{r} \sum_{j=1}^{r} \text{CV}_{(k)}^{(j)}.
$$
Повторяющаяся кросс‑валидация позволяет не только получить более стабильную точечную оценку, но и оценить её разброс, анализируя распределение результатов по повторностям. В `sklearn` используется `RepeatedKFold` (или `RepeatedStratifiedKFold`).

### 5. Практическая демонстрация на Python

Продемонстрируем поведение различных схем кросс‑валидации на примере классификации рака молочной железы.

#### 5.1. Сравнение k‑fold, LOOCV и repeated k‑fold

```python
# Загрузка данных
X, y = load_breast_cancer(return_X_y=True)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Модель
model = LogisticRegression(max_iter=10000)

# Схемы кросс-валидации
schemes = {
    '5-fold': KFold(n_splits=5, shuffle=True, random_state=42),
    '10-fold': KFold(n_splits=10, shuffle=True, random_state=42),
    'LOOCV': LeaveOneOut(),
    'Repeated 5-fold (10 reps)': RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)
}

results = {}
for name, cv in schemes.items():
    scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy')
    results[name] = scores
    print(f"{name:25s}: mean={scores.mean():.4f}, std={scores.std():.4f}")
```

Результаты (типичный вывод):

```
5-fold                   : mean=0.9789, std=0.0095
10-fold                  : mean=0.9789, std=0.0131
LOOCV                    : mean=0.9789, std=0.1437
Repeated 5-fold (10 reps): mean=0.9790, std=0.0102
```

Видно, что средняя accuracy практически одинакова, но стандартное отклонение по фолдам у LOOCV заметно выше, что отражает коррелированность оценок.

#### 5.2. Боксплот распределений accuracy

```python
plt.figure(figsize=(10, 5))
data_for_plot = [results['5-fold'], results['10-fold'],
                 results['Repeated 5-fold (10 reps)']]
labels = ['5-fold', '10-fold', 'Repeated 5-fold']
plt.boxplot(data_for_plot, labels=labels)
plt.ylabel('Accuracy')
plt.title('Распределение accuracy по фолдам/повторам')
plt.grid(axis='y')
plt.show()
```

(В случае LOOCV боксплот строить нецелесообразно из-за большого числа фолдов, равного $n$.)

#### 5.3. Демонстрация стратификации

Создадим синтетический набор данных с сильным дисбалансом классов (5% миноритарного класса) и сравним обычный KFold со стратифицированным.

```python
X_imb, y_imb = make_classification(n_samples=200, n_features=5, weights=[0.95, 0.05],
                                   random_state=42)
print("Распределение классов:", np.bincount(y_imb))

# Обычный KFold
kf = KFold(n_splits=5, shuffle=True, random_state=0)
print("\nОбычный KFold (миноритарный класс в тесте):")
for i, (train_idx, test_idx) in enumerate(kf.split(X_imb, y_imb)):
    test_labels = y_imb[test_idx]
    print(f"  Фолд {i}: всего {len(test_labels)}, минор. класс {np.sum(test_labels == 1)}")

# Стратифицированный KFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
print("\nStratifiedKFold (миноритарный класс в тесте):")
for i, (train_idx, test_idx) in enumerate(skf.split(X_imb, y_imb)):
    test_labels = y_imb[test_idx]
    print(f"  Фолд {i}: всего {len(test_labels)}, минор. класс {np.sum(test_labels == 1)}")
```

Пример вывода:

```
Распределение классов: [190  10]
Обычный KFold (миноритарный класс в тесте):
  Фолд 0: всего 40, минор. класс 4
  Фолд 1: всего 40, минор. класс 0
  Фолд 2: всего 40, минор. класс 2
  Фолд 3: всего 40, минор. класс 2
  Фолд 4: всего 40, минор. класс 2
StratifiedKFold (миноритарный класс в тесте):
  Фолд 0: всего 40, минор. класс 2
  Фолд 1: всего 40, минор. класс 2
  Фолд 2: всего 40, минор. класс 2
  Фолд 3: всего 40, минор. класс 2
  Фолд 4: всего 40, минор. класс 2
```

Обычный KFold допустил появление фолда (№1) без миноритарного класса, что сделало бы оценку качества на этом фолде некорректной. Стратификация гарантирует постоянное присутствие миноритарного класса во всех тестовых блоках.

---

## Вопросы для самопроверки

### Для начинающих

1. Что такое k‑fold кросс‑валидация? Как она работает и для чего применяется?
2. Чем LOOCV отличается от 10‑fold кросс‑валидации с точки зрения объёма обучающих данных и числа итераций?
3. Зачем нужна стратифицированная кросс‑валидация? К каким проблемам может привести её отсутствие при несбалансированных классах?
4. Какую цель преследует повторяющаяся кросс‑валидация (Repeated k‑fold)?
5. Как следует интерпретировать стандартное отклонение accuracy, вычисленной по фолдам? Можно ли ему полностью доверять?

### Для профессионалов

1. Докажите, что для линейной регрессии LOOCV даёт практически несмещённую оценку ожидаемой ошибки (используя выражение через $h_{ii}$).
2. Объясните, почему оценки ошибок на разных фолдах k‑fold кросс‑валидации коррелированы, и как это влияет на стандартную ошибку среднего.
3. Приведите формулу для стандартной ошибки кросс‑валидации с учётом корреляции между фолдами (используя общую формулу дисперсии суммы зависимых величин).
4. Сравните смещение оценки k‑fold CV при малых и больших $k$ и объясните, почему на практике чаще используют $k = 5$ или $k = 10$, а не LOOCV.

Ниже представлено продолжение главы — раздел, посвящённый кросс‑валидации при зависимых данных, с акцентом на временные ряды и пространственно‑временные структуры.

---

## Часть III. Кросс‑валидация для временных рядов и зависимых данных

Все рассмотренные ранее схемы кросс‑валидации предполагают, что наблюдения независимы и одинаково распределены. Однако во многих прикладных задачах — от прогнозирования продаж до анализа медицинских записей — данные обладают внутренней зависимостью. Игнорирование этой структуры приводит к катастрофически оптимистичным оценкам качества модели и ложному чувству прогресса.

### 1. Проблема зависимых данных

Стандартная k‑fold кросс‑валидация случайным образом перемешивает индексы наблюдений. Если данные представляют собой временной ряд или содержат группы взаимосвязанных записей, такое перемешивание создаёт **утечку информации** из будущего в прошлое или между сильно скоррелированными объектами, оказавшимися в обучении и тесте. Модель начинает «жульничать»: она использует закономерности, которые в реальной эксплуатации будут недоступны, и демонстрирует неоправданно высокую точность.

Фундаментальный принцип при работе с временными рядами и другими зависимыми данными: **обучающая выборка должна предшествовать тестовой во времени или, в более общем смысле, быть строго отделённой от неё по структуре зависимости**. Только тогда оценка ошибки будет отражать поведение модели в условиях реального развёртывания.

### 2. Кросс‑валидация на временных рядах

#### 2.1. Скользящий контроль (Time Series Split)

Наиболее естественная схема для временных рядов — последовательное наращивание обучающей выборки. Предположим, что наблюдения упорядочены по времени: $y_1, y_2, \dots, y_n$, и каждому моменту $t$ соответствует вектор признаков $\mathbf{x}_t$. В классическом **TimeSeriesSplit** задаётся минимальный размер начальной обучающей выборки $n_{\text{train}}$ и число разбиений $k$. Процедура формирует $k$ пар:

- **Фолд 1:** обучаемся на $y_1, \dots, y_{n_{\text{train}}}$, тестируем на $y_{n_{\text{train}}+1}, \dots, y_{n_{\text{train}}+h}$;
- **Фолд 2:** обучаемся на $y_1, \dots, y_{n_{\text{train}}+h}$, тестируем на следующем блоке и т.д.

Формально, для $i = 0, \dots, k-1$:
$$
\mathcal{D}_{\text{train}}^{(i)} = \{1, \dots, n_{\text{train}} + i \cdot \Delta\}, \qquad
\mathcal{D}_{\text{test}}^{(i)} = \{n_{\text{train}} + i \cdot \Delta + 1, \dots, n_{\text{train}} + (i+1) \cdot \Delta\},
$$
где $\Delta$ — размер тестового блока (обычно выбирается равным $h$ или так, чтобы заполнить оставшиеся данные). В `sklearn` это реализовано классом `TimeSeriesSplit`, где параметр `n_splits` задаёт количество разбиений, а `max_train_size` может ограничивать размер обучающей выборки.

Важно отметить, что обучающая выборка с каждым шагом растёт, и модель на последних итерациях обучается на значительно большем объёме данных, чем на первых. Это приводит к смещённой оценке ожидаемой ошибки — она усредняется по моделям, обученным на разных объёмах истории. Однако именно такая схема имитирует реальный сценарий: когда модель запускается в эксплуатацию, она тоже сначала обучается на короткой истории, а затем по мере накопления данных периодически переобучается.

#### 2.2. Скользящий контроль с фиксированным окном

Альтернативный подход — **скользящее окно** (rolling window) с постоянным размером обучающей выборки $w$. Здесь на каждом шаге мы берём последние $w$ наблюдений перед тестовым блоком:
$$
\mathcal{D}_{\text{train}}^{(i)} = \{i+1, \dots, i+w\}, \qquad
\mathcal{D}_{\text{test}}^{(i)} = \{i+w+1, \dots, i+w+h\}.
$$
Этот метод лучше подходит для стационарных процессов, где не предполагается накопление исторических данных, а модель должна оперативно адаптироваться к последним изменениям. В `sklearn` для этого можно использовать `TimeSeriesSplit` с параметром `max_train_size=w`.

#### 2.3. Смещение оценки и дисперсия

Случайная k‑fold кросс‑валидация на временном ряду даёт смещённую вниз (оптимистичную) оценку ошибки, поскольку тестовые точки часто оказываются окружены обучающими, и модель может эксплуатировать временную близость. Напротив, TimeSeriesSplit даёт пессимистичную оценку (смещение вверх) из‑за обучения на укороченной истории. Выбор схемы определяется целью: если нужно оценить качество модели в условиях, максимально приближенных к реальности, TimeSeriesSplit предпочтительнее. Если же интерес представляет предельная точность при большом объёме данных, можно рассматривать специальные корректировки.

> **Углублённый взгляд: смещение оценки при случайном k‑fold для AR(1)**
>
> Рассмотрим стационарный авторегрессионный процесс первого порядка:
> $$
> y_t = \phi y_{t-1} + \varepsilon_t, \quad |\phi| < 1, \quad \varepsilon_t \sim \mathcal{N}(0, \sigma^2).
> $$
> Предположим, мы строим модель $\hat{y}_t = \hat{\phi} y_{t-1}$, оценивая $\hat{\phi}$ методом наименьших квадратов. При случайном k‑fold некоторые пары $(y_t, y_{t-1})$ окажутся в обучении, а их соседи — в тесте. Модель может «запомнить» локальные флуктуации, что приводит к завышенной оценке предсказательной силы. В пределе, если $\phi \to 1$ (случайное блуждание), корреляция между соседними наблюдениями максимальна, и смещение становится особенно сильным. TimeSeriesSplit гарантирует, что все обучающие наблюдения предшествуют тестовым, устраняя этот источник смещения.

### 3. Блоковая кросс‑валидация для зависимых данных

Не всякая зависимость имеет временную природу. Часто данные естественным образом группируются: пациенты в медицинских исследованиях, пользователи в рекомендательных системах, датчики в промышленности. Наблюдения внутри одной группы коррелированы, и их попадание в разные фолды также приводит к утечке информации.

#### 3.1. Group k‑fold

В **GroupKFold** задаётся вектор групп $g_i$ для каждого наблюдения. Алгоритм гарантирует, что все записи с одинаковым значением $g$ попадут либо только в обучение, либо только в тест. Фолды формируются на уровне групп: группы случайно разбиваются на $k$ блоков, а затем эти блоки преобразуются в индексы наблюдений. В `sklearn` класс `GroupKFold` реализует эту логику.

#### 3.2. Стратификация по группам

Если внутри групп имеется дисбаланс классов, используют `StratifiedGroupKFold` (доступен в `sklearn` начиная с версии 1.0). Он одновременно сохраняет распределение классов и не допускает смешивания групп.

Групповая кросс‑валидация критически важна, например, при прогнозировании заболеваний: записи одного пациента должны быть либо в обучении, либо в тесте, иначе модель выучит индивидуальные особенности пациента, а не общие диагностические закономерности.

### 4. Пространственно‑временные данные

Данные, распределённые в пространстве и времени (спутниковые снимки, метеорологические наблюдения, геолокация), обладают одновременно временной и пространственной автокорреляцией. Соседние пиксели или близкие метеостанции сильно скоррелированы. Случайное разбиение таких данных приводит к тому, что обучающие и тестовые точки находятся в непосредственной близости, и модель легко интерполирует, не обладая способностью к пространственному обобщению.

Правильный подход — **пространственная кросс‑валидация**, при которой тестовые блоки выбираются географически удалёнными от обучающих. Например, всю область разбивают на непересекающиеся пространственные блоки (сетку), и на каждой итерации один или несколько блоков объявляются тестовыми, а остальные — обучающими. Это позволяет проверить, способна ли модель экстраполировать знания на новые регионы, что особенно важно в экологии, климатологии и сельском хозяйстве.

В простейшем случае можно использовать `GroupKFold`, назначив каждой пространственной ячейке уникальный идентификатор группы. Для данных, где важна и временна́я структура, комбинируют временной и пространственный принципы: тестовый период должен следовать за обучающим, и тестовые локации не должны пересекаться с обучающими.

### 5. Демонстрация на Python

Проиллюстрируем опасности игнорирования временной структуры и работу специализированных методов.

#### 5.1. Генерация синтетического временного ряда

Создадим авторегрессионный процесс AR(1) с $\phi=0.9$, добавим трендовую компоненту и обучим линейную регрессию с признаком «время».

```python
np.random.seed(0)
n = 200
time_idx = np.arange(n).reshape(-1, 1)
phi = 0.9
eps = np.random.randn(n) * 0.5
y = np.zeros(n)
for t in range(1, n):
    y[t] = phi * y[t-1] + eps[t] + 0.02 * t  # лёгкий тренд
```

#### 5.2. Сравнение обычного KFold и TimeSeriesSplit

```python
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.linear_model import LinearRegression

model = LinearRegression()

# Обычный 5-fold (перемешанный)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kf = cross_val_score(model, time_idx, y, cv=kf, scoring='neg_mean_squared_error')
rmse_kf = np.sqrt(-scores_kf)

# TimeSeriesSplit (5 разбиений)
tscv = TimeSeriesSplit(n_splits=5)
scores_ts = cross_val_score(model, time_idx, y, cv=tscv, scoring='neg_mean_squared_error')
rmse_ts = np.sqrt(-scores_ts)

print("KFold RMSE (mean ± std): {:.3f} ± {:.3f}".format(rmse_kf.mean(), rmse_kf.std()))
print("TimeSeriesSplit RMSE (mean ± std): {:.3f} ± {:.3f}".format(rmse_ts.mean(), rmse_ts.std()))
```

Типичный вывод демонстрирует, что KFold даёт существенно заниженную ошибку (модель как будто «предсказывает» почти идеально), тогда как TimeSeriesSplit показывает более высокую и реалистичную ошибку, поскольку тестовые точки всегда находятся за пределами обучающего периода.

#### 5.3. Визуализация разбиений

Покажем, как распределены индексы обучающих и тестовых блоков.

```python
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# KFold
cv_kf = KFold(n_splits=5, shuffle=True, random_state=42)
axes[0].set_title('KFold (shuffled)')
for i, (train_idx, test_idx) in enumerate(cv_kf.split(time_idx)):
    axes[0].scatter(train_idx, [i]*len(train_idx), c='blue', s=5, label='train' if i==0 else "")
    axes[0].scatter(test_idx, [i]*len(test_idx), c='red', s=5, label='test' if i==0 else "")
axes[0].set_xlabel('Индекс наблюдения')
axes[0].set_ylabel('Фолд')
axes[0].legend()

# TimeSeriesSplit
cv_ts = TimeSeriesSplit(n_splits=5)
axes[1].set_title('TimeSeriesSplit')
for i, (train_idx, test_idx) in enumerate(cv_ts.split(time_idx)):
    axes[1].scatter(train_idx, [i]*len(train_idx), c='blue', s=5)
    axes[1].scatter(test_idx, [i]*len(test_idx), c='red', s=5)
axes[1].set_xlabel('Индекс наблюдения')
axes[1].legend()
plt.tight_layout()
plt.show()
```

Рисунок наглядно покажет, что при KFold тестовые точки разбросаны по всему временному диапазону, тогда как при TimeSeriesSplit они идут последовательными блоками, строго после обучающих.

#### 5.4. Пример групповой кросс‑валидации

Смоделируем данные пациентов: у каждого пациента по несколько визитов, и целевая переменная коррелирует внутри пациента.

```python
n_patients = 20
records_per_patient = 5
total = n_patients * records_per_patient
patient_ids = np.repeat(np.arange(n_patients), records_per_patient)
X_group = np.random.randn(total, 3)
# целевая переменная с групповым эффектом
group_effect = np.random.randn(n_patients) * 2.0
y_group = X_group[:, 0] * 1.5 + group_effect[patient_ids] + np.random.randn(total) * 0.5

from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=5)
fold_group_scores = []
for train_idx, test_idx in gkf.split(X_group, y_group, groups=patient_ids):
    # проверяем, что ID пациентов в обучении и тесте не пересекаются
    train_patients = set(patient_ids[train_idx])
    test_patients = set(patient_ids[test_idx])
    assert train_patients.isdisjoint(test_patients)
    model = LinearRegression()
    model.fit(X_group[train_idx], y_group[train_idx])
    pred = model.predict(X_group[test_idx])
    fold_group_scores.append(np.sqrt(np.mean((pred - y_group[test_idx])**2)))
print("GroupKFold RMSE по фолдам:", np.mean(fold_group_scores))
```

Этот код демонстрирует, что ни один пациент не появляется одновременно в обучении и тесте.

---

## Вопросы для самопроверки

### Для начинающих

1. Почему обычную k‑fold кросс‑валидацию нельзя напрямую применять к временным рядам?
2. Как работает TimeSeriesSplit? Чем он принципиально отличается от KFold?
3. Зачем нужна групповая кросс‑валидация (GroupKFold)? Приведите пример данных, где она необходима.
4. Что такое утечка данных (data leakage) и как она проявляется при работе с зависимыми выборками?

### Для профессионалов

1. Выведите, что для AR(1) процесса оценка ошибки при случайном k‑fold смещена вниз. Какие параметры процесса усиливают это смещение?
2. Предложите стратегию выбора размера обучающего окна $w$ в скользящем контроле с фиксированным окном. Как этот выбор влияет на смещение и дисперсию оценки?
3. Сравните дисперсию оценки при блоковой кросс‑валидации (например, GroupKFold) и при обычной k‑fold. Почему дисперсия в блоковой схеме может быть выше?
4. Опишите, как модифицировать кросс‑валидацию для пространственных данных, чтобы избежать недооценки ошибки из‑за пространственной автокорреляции. Какие метрики близости (расстояния) можно использовать для определения блоков?

Ниже представлена следующая часть главы — о методах выбора гиперпараметров. Мы продолжаем использовать библиотеки, импортированные ранее, и добавляем несколько новых.

---

## Часть IV. Выбор гиперпараметров: grid search, random search и байесовская оптимизация

После того как мы научились корректно оценивать качество модели с помощью кросс‑валидации, встаёт вопрос: как найти такие значения гиперпараметров, при которых это качество максимально? Гиперпараметры — это параметры модели, не настраиваемые по данным напрямую (например, коэффициент регуляризации $\lambda$, параметр $C$ в SVM, ширина ядра $\gamma$). Их выбор традиционно опирается на перебор с валидацией, но наивный перебор быстро становится неосуществимым. Мы последовательно рассмотрим три семейства методов — от простого поиска по сетке до байесовской оптимизации, — и покажем их сравнительную эффективность.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV,
                                     cross_val_score, KFold)
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from scipy.stats import uniform, loguniform
from scipy.optimize import minimize
```

### 1. Постановка задачи выбора гиперпараметров

Пусть модель задана семейством функций $\mathcal{M}_{\boldsymbol{\lambda}}$, где $\boldsymbol{\lambda} \in \Lambda$ — вектор гиперпараметров. Истинное качество модели характеризуется ожидаемой ошибкой $\text{Err}(\boldsymbol{\lambda})$, которую на практике оценивают с помощью кросс‑валидации:
$$
\widehat{\text{Err}}(\boldsymbol{\lambda}) = \frac{1}{K} \sum_{i=1}^{K} \text{Err}_{\text{fold}_i}(\boldsymbol{\lambda}).
$$
Задача выбора гиперпараметров сводится к минимизации этой оценки:
$$
\boldsymbol{\lambda}^* = \arg\min_{\boldsymbol{\lambda} \in \Lambda} \widehat{\text{Err}}(\boldsymbol{\lambda}).
$$
Основная трудность в том, что каждое вычисление $\widehat{\text{Err}}(\boldsymbol{\lambda})$ требует обучения модели (возможно, нескольких), что может занимать минуты или часы. Поэтому необходимо спланировать процесс перебора так, чтобы за ограниченное число обращений к целевой функции найти как можно лучшее решение.

### 2. Grid search (поиск по сетке)

Самый прямолинейный подход — задать для каждого гиперпараметра конечный набор значений и перебрать все возможные комбинации. Например, для SVM с RBF‑ядром это может быть:
$$
C \in \{0.1, 1, 10, 100\}, \quad \gamma \in \{0.001, 0.01, 0.1, 1\}.
$$
Общее число точек сетки равно произведению числа значений по каждому параметру. Достоинство grid search в том, что он гарантированно находит минимум на заданной сетке. Недостаток — «проклятие размерности»: при $d$ гиперпараметрах и $m$ значениях на каждый требуется $m^d$ вычислений. Уже при $d=5$, $m=10$ это $100\,000$ запусков, что часто неприемлемо.

Тем не менее для небольшого числа параметров grid search остаётся разумным выбором. Рекомендуется проводить его в два этапа: сначала грубая сетка, затем более мелкая вокруг лучших найденных точек.

#### Демонстрация на Python

```python
# Загрузка данных
X, y = load_breast_cancer(return_X_y=True)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Параметры сетки
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1]
}
grid = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, scoring='accuracy', return_train_score=True)
grid.fit(X_scaled, y)

print("Лучшие параметры:", grid.best_params_)
print("Лучшая CV accuracy: {:.4f}".format(grid.best_score_))
```

Построим heatmap точности на кросс‑валидации в зависимости от $C$ и $\gamma$.

```python
scores = grid.cv_results_['mean_test_score'].reshape(len(param_grid['C']), len(param_grid['gamma']))

plt.figure(figsize=(8, 6))
plt.imshow(scores, interpolation='nearest', cmap='viridis')
plt.colorbar(label='Accuracy')
plt.xticks(range(len(param_grid['gamma'])), param_grid['gamma'], rotation=45)
plt.yticks(range(len(param_grid['C'])), param_grid['C'])
plt.xlabel('gamma')
plt.ylabel('C')
plt.title('Grid Search: CV Accuracy')
for i in range(len(param_grid['C'])):
    for j in range(len(param_grid['gamma'])):
        plt.text(j, i, f'{scores[i,j]:.3f}', ha='center', va='center', color='w' if scores[i,j] < 0.95 else 'k')
plt.show()
```

### 3. Random search (случайный поиск)

Вместо систематического перебора можно случайно выбрать $N$ комбинаций гиперпараметров из заданных распределений. Bergstra и Bengio (2012) показали, что этот подход часто оказывается эффективнее grid search, особенно когда лишь небольшое число гиперпараметров действительно влияет на результат.

Формальное обоснование. Пусть $p$ — доля объёма пространства $\Lambda$, в котором значение целевой функции превосходит некоторый порог (т.е. «хорошая» область). При случайном выборе $N$ точек вероятность того, что хотя бы одна из них окажется в хорошей области, равна
$$
P(\text{успех}) = 1 - (1-p)^N.
$$
Эта вероятность не зависит от размерности пространства. В grid search, чтобы сохранить фиксированную плотность покрытия, необходимо экспоненциально увеличивать число точек с ростом размерности. Следовательно, при одинаковом бюджете $N$ случайный поиск с высокой вероятностью находит решение не хуже, а часто и лучше, чем поиск по сетке.

> **Углублённый взгляд: доказательство гарантии random search**
>
> Пусть $\lambda$ выбирается равномерно из $\Lambda$ (или из заданного распределения). Вероятность попасть в область $A$ с мерой $p$ равна $p$. Для $N$ независимых испытаний вероятность ни разу не попасть в $A$ равна $(1-p)^N$. Следовательно, вероятность хотя бы одного попадания равна $1 - (1-p)^N$. Для любого заданного уровня уверенности $1-\delta$ необходимо $N \ge \frac{\log \delta}{\log(1-p)}$ испытаний. Эта граница не содержит размерности $d$. При grid search для покрытия $d$-мерного куба с шагом $1/m$ требуется $m^d$ точек; чтобы доля покрытия оставалась постоянной при росте $d$, $m$ должно расти, и число точек экспоненциально увеличивается. Таким образом, random search экспоненциально эффективнее grid search в высоких размерностях.

#### Демонстрация random search

```python
param_distributions = {
    'C': loguniform(1e-2, 1e2),      # логарифмически равномерное
    'gamma': loguniform(1e-4, 1e0)
}
random_search = RandomizedSearchCV(
    SVC(kernel='rbf'), param_distributions, n_iter=30, cv=5,
    scoring='accuracy', random_state=42, n_jobs=-1
)
random_search.fit(X_scaled, y)

print("Лучшие параметры (random):", random_search.best_params_)
print("Лучшая CV accuracy: {:.4f}".format(random_search.best_score_))
```

Сравним время и качество с grid search (можно построить график результатов).

### 4. Байесовская оптимизация

Random search не использует информацию о ранее испытанных точках. Байесовская оптимизация строит вероятностную модель (суррогатную модель) зависимости $\widehat{\text{Err}}(\boldsymbol{\lambda})$ от гиперпараметров и на каждой итерации выбирает следующую точку, максимизируя **функцию приобретения** (acquisition function). Функция приобретения балансирует между исследованием областей с высокой неопределённостью (exploration) и эксплуатацией областей с низким предсказанным значением ошибки (exploitation). В качестве суррогатной модели чаще всего применяют **гауссовский процесс** (GP).

**Гауссовский процесс** задаёт априорное распределение над функциями. После наблюдения $t$ точек $(\boldsymbol{\lambda}_i, e_i)$ GP даёт апостериорное распределение: для любого $\boldsymbol{\lambda}$ предсказание имеет вид $\mathcal{N}(\mu(\boldsymbol{\lambda}), \sigma^2(\boldsymbol{\lambda}))$. Ожидаемое улучшение (Expected Improvement, EI) для минимизации определяется как
$$
\text{EI}(\boldsymbol{\lambda}) = \mathbb{E}\bigl[\max(0, e_{\text{best}} - f(\boldsymbol{\lambda}))\bigr],
$$
где $e_{\text{best}}$ — лучшее наблюдённое значение. Это выражение имеет аналитический вид:
$$
\text{EI}(\boldsymbol{\lambda}) = (e_{\text{best}} - \mu(\boldsymbol{\lambda}))\,\Phi(Z) + \sigma(\boldsymbol{\lambda})\,\phi(Z), \quad
Z = \frac{e_{\text{best}} - \mu(\boldsymbol{\lambda})}{\sigma(\boldsymbol{\lambda})},
$$
где $\Phi$ и $\phi$ — функция распределения и плотность стандартного нормального закона. Следующая точка выбирается как $\arg\max \text{EI}(\boldsymbol{\lambda})$.

Алгоритм:
1. Выбрать несколько начальных точек (например, случайно), вычислить ошибку.
2. Пока не исчерпан бюджет:
   - Обучить GP на всех наблюдённых точках.
   - Найти $\boldsymbol{\lambda}_{\text{new}} = \arg\max \text{EI}(\boldsymbol{\lambda})$ с помощью численной оптимизации (например, L‑BFGS).
   - Вычислить $\widehat{\text{Err}}(\boldsymbol{\lambda}_{\text{new}})$ и добавить пару в обучающую выборку GP.

#### Ручная реализация байесовской оптимизации для одного гиперпараметра

Для наглядности рассмотрим одномерный случай — подбор параметра $C$ SVM (gamma фиксирована). Целевая функция — accuracy на кросс‑валидации (или отрицательная accuracy, чтобы минимизировать ошибку). Будем использовать `GaussianProcessRegressor` из `sklearn`.

```python
# Одномерный пример: подбор C для SVM (gamma=0.01)
def cv_error(C):
    # C - float, возвращает -accuracy (минимизируем)
    model = SVC(C=C, kernel='rbf', gamma=0.01)
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
    return -np.mean(scores)

# Границы поиска
bounds = np.array([[1e-2, 1e2]])
np.random.seed(123)

# Начальные точки
n_init = 3
X_init = np.random.uniform(bounds[0,0], bounds[0,1], size=(n_init, 1))
y_init = np.array([cv_error(C) for C in X_init.ravel()])

# GP с ядром RBF
kernel = C(1.0, (1e-3, 1e3)) * RBF(10, (1e-2, 1e2))
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, alpha=1e-6)

# Функция Expected Improvement
def expected_improvement(X, gp, y_best):
    X = np.atleast_2d(X)
    mu, sigma = gp.predict(X, return_std=True)
    with np.errstate(divide='warn'):
        Z = (y_best - mu) / sigma
        ei = (y_best - mu) * sp.stats.norm.cdf(Z) + sigma * sp.stats.norm.pdf(Z)
        ei[sigma == 0.0] = 0.0
    return -ei  # отрицательное, чтобы использовать minimize

# Байесовская оптимизация
n_iter = 10
X_obs = np.array(X_init)
y_obs = np.array(y_init)
for i in range(n_iter):
    gp.fit(X_obs, y_obs)
    y_best = np.min(y_obs)
    # минимизация отрицательного EI
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=np.random.uniform(bounds[0,0], bounds[0,1]),
                   bounds=bounds, method='L-BFGS-B')
    next_C = res.x[0]
    next_error = cv_error(next_C)
    X_obs = np.vstack([X_obs, [next_C]])
    y_obs = np.append(y_obs, next_error)
    print(f"Итерация {i+1}: C={next_C:.4f}, ошибка={-next_error:.4f}")

best_idx = np.argmin(y_obs)
print(f"Лучшее найденное C={X_obs[best_idx][0]:.4f}, accuracy={-y_obs[best_idx]:.4f}")
```

Визуализируем процесс: истинная кривая ошибки (полученная на мелкой сетке), предсказания GP, функцию приобретения.

```python
C_range = np.logspace(-2, 2, 200)
true_errors = np.array([cv_error(c) for c in C_range])
X_plot = C_range.reshape(-1, 1)

gp.fit(X_obs, y_obs)
mu, sigma = gp.predict(X_plot, return_std=True)
y_best = np.min(y_obs)
ei = -expected_improvement(X_plot, gp, y_best)  # восстановим знак

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
ax1.plot(C_range, -true_errors, 'k-', label='Истинная accuracy')
ax1.plot(X_obs.ravel(), -y_obs, 'r.', markersize=10, label='Наблюдения')
ax1.fill_between(C_range, -(mu - 1.96*sigma), -(mu + 1.96*sigma), alpha=0.3, label='GP 95% CI')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax2.plot(C_range, ei, 'b-', label='Expected Improvement')
ax2.set_xlabel('C')
ax2.set_ylabel('EI')
ax2.legend()
plt.xscale('log')
plt.suptitle('Байесовская оптимизация')
plt.show()
```

Библиотеки типа Optuna или scikit‑optimize автоматизируют этот процесс, но ручная реализация раскрывает суть метода.

### 5. Сравнение методов

Для объективного сравнения построим кривые сходимости — лучшее найденное значение accuracy в зависимости от числа итераций (обучений модели). Будем использовать ту же задачу SVM с двумя гиперпараметрами ($C$, $\gamma$). Grid search здесь не совсем честен, так как требует полного перебора; мы можем рассматривать его как случайный выбор точек из фиксированной сетки по мере прохождения.

```python
# Сравнение random и bayesian (простейшая реализация для двух параметров)
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

def cv_error_2d(C, gamma):
    model = SVC(C=C, kernel='rbf', gamma=gamma)
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
    return -np.mean(scores)

# Random search: 30 итераций
np.random.seed(42)
random_C = loguniform.rvs(1e-2, 1e2, size=30)
random_gamma = loguniform.rvs(1e-4, 1e0, size=30)
random_results = []
best_rand = -np.inf
for C, g in zip(random_C, random_gamma):
    err = cv_error_2d(C, g)
    best_rand = max(best_rand, -err)
    random_results.append(best_rand)

# Байесовская оптимизация для двух параметров (упрощённый вариант с GP)
bounds_2d = np.array([[1e-2, 1e2], [1e-4, 1e0]])
n_init = 5
X_init_2d = np.random.uniform(bounds_2d[:,0], bounds_2d[:,1], size=(n_init, 2))
y_init_2d = np.array([cv_error_2d(x[0], x[1]) for x in X_init_2d])

kernel_2d = C(1.0) * RBF([1.0, 1.0], (1e-2, 1e2))
gp_2d = GaussianProcessRegressor(kernel=kernel_2d, n_restarts_optimizer=10, alpha=1e-6)

X_obs_2d = np.array(X_init_2d)
y_obs_2d = np.array(y_init_2d)
bayes_results = []
best_bayes = -np.inf
for i in range(30):
    gp_2d.fit(X_obs_2d, y_obs_2d)
    y_best = np.min(y_obs_2d)
    # Оптимизация EI по двум параметрам
    def ei_2d(x):
        return expected_improvement([x], gp_2d, y_best)
    res = minimize(lambda x: expected_improvement(x, gp_2d, y_best),
                   x0=np.random.uniform(bounds_2d[:,0], bounds_2d[:,1]),
                   bounds=bounds_2d, method='L-BFGS-B')
    next_x = res.x
    next_err = cv_error_2d(next_x[0], next_x[1])
    X_obs_2d = np.vstack([X_obs_2d, next_x])
    y_obs_2d = np.append(y_obs_2d, next_err)
    best_bayes = max(best_bayes, -next_err)
    bayes_results.append(best_bayes)

plt.figure(figsize=(8,5))
plt.plot(range(1, 31), random_results, label='Random Search')
plt.plot(range(1, 31), bayes_results, label='Bayesian Optimization')
plt.xlabel('Число итераций')
plt.ylabel('Лучшая accuracy')
plt.legend()
plt.grid()
plt.title('Сравнение сходимости')
plt.show()
```

Результаты обычно показывают, что байесовская оптимизация достигает более высокого качества за меньшее число итераций, особенно когда целевая функция multimodal и имеет значительные области плохого качества.

**Когда что выбирать:**
- **Grid search** — при 1–2 параметрах и быстрой оценке модели, когда важна систематичность и интерпретируемость.
- **Random search** — при умеренной размерности (до ~10) и ограниченном бюджете; служит хорошим baseline.
- **Байесовская оптимизация** — при дорогих вычислениях и размерности до 20, когда требуется максимальная эффективность использования каждого эксперимента.

---

## Вопросы для самопроверки

### Для начинающих

1. Чем случайный поиск (random search) принципиально лучше поиска по сетке при большом числе гиперпараметров?
2. Что такое суррогатная модель в байесовской оптимизации и зачем она нужна?
3. Почему байесовская оптимизация может требовать меньше вычислений, чем случайный поиск?
4. Какую роль играет функция приобретения (acquisition function)? Приведите пример.

### Для профессионалов

1. Докажите, что для случайного поиска вероятность попасть в «хорошую» область не зависит от размерности пространства. Объясните, почему это даёт экспоненциальное преимущество над grid search.
2. Выпишите математическое определение Expected Improvement и поясните смысл каждого члена. Как оно балансирует exploration и exploitation?
3. Как гауссовский процесс позволяет вычислить неопределённость предсказания? Какие параметры ядра влияют на гладкость суррогатной модели?
4. Какие проблемы масштабирования возникают у байесовской оптимизации при большом числе гиперпараметров (более 20) и как их можно преодолеть?

Ниже представлена заключительная часть главы, объединяющая nested CV, анализ ансамблей и итоговые рекомендации.

---

## Часть V. Nested CV, bias‑variance для ансамблей и итоговое резюме

Мы научились оценивать качество модели и выбирать гиперпараметры. Однако между этими шагами существует тонкая, но принципиально важная взаимосвязь: если использовать одни и те же данные и для выбора модели, и для итоговой оценки её качества, мы получим оптимистично смещённую оценку. В этой части мы обсудим, как избежать этой ловушки с помощью вложенной кросс‑валидации, поймём, почему ансамбли моделей работают так хорошо, и подведём итог всему изученному материалу.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (KFold, StratifiedKFold, cross_val_score,
                                     GridSearchCV, RandomizedSearchCV)
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
```

### 1. Проблема смещения при выборе модели

Предположим, мы настраиваем гиперпараметры SVM с помощью 10‑fold кросс‑валидации и выбираем конфигурацию, показавшую наилучшее среднее качество на этих 10 фолдах. Затем мы берём эту лучшую модель, обучаем её на всей выборке и сообщаем полученную на кросс‑валидации accuracy как оценку её обобщающей способности. Это **некорректно**: мы уже использовали валидационные фолды для выбора модели, и итоговая оценка оказывается смещённой вниз (излишне оптимистичной). По сути, валидационная выборка «подтекает» в процесс выбора модели, и оценка перестаёт быть честной.

Величина этого смещения может быть значительной, особенно если пространство гиперпараметров велико, а данных мало. Настоящая обобщающая способность процедуры «обучение + выбор гиперпараметров» должна оцениваться на данных, которые ни на каком этапе не использовались ни для обучения, ни для выбора модели.

Решение — **вложенная кросс‑валидация** (nested cross‑validation).

### 2. Nested CV на практике

Идея вложенной кросс‑валидации состоит в разделении ролей: внешний цикл отвечает за итоговую оценку качества, внутренний — за выбор гиперпараметров. Алгоритм выглядит следующим образом:

1. Разбить все данные на $K_{\text{out}}$ внешних фолдов.
2. Для каждого внешнего фолда $i$:
   - Объявить его тестовым множеством $\mathcal{D}_{\text{test}}^{(i)}$, оставшиеся $K_{\text{out}}-1$ фолдов — внешней обучающей выборкой $\mathcal{D}_{\text{train}}^{(i)}$.
   - На $\mathcal{D}_{\text{train}}^{(i)}$ выполнить внутреннюю кросс‑валидацию (например, $K_{\text{in}}$-fold) или другой метод подбора гиперпараметров. Найти лучшие гиперпараметры $\boldsymbol{\lambda}_i^*$ по внутренней валидации.
   - Обучить модель на всей $\mathcal{D}_{\text{train}}^{(i)}$ с гиперпараметрами $\boldsymbol{\lambda}_i^*$ и вычислить ошибку на $\mathcal{D}_{\text{test}}^{(i)}$.
3. Итоговая оценка качества — среднее значение ошибки по всем $K_{\text{out}}$ внешним тестовым фолдам.

Таким образом, каждый внешний тестовый фолд ни разу не участвовал в выборе гиперпараметров для модели, которая на нём оценивается. Получаемая оценка является практически несмещённой для полной процедуры «внутренний CV + обучение на всех доступных обучающих данных».

#### Демонстрация на Python

Сравним обычную (наивную) оценку через GridSearchCV и честную оценку через nested CV на задаче SVM с RBF‑ядром.

```python
# Загрузим данные и стандартизируем
X, y = load_breast_cancer(return_X_y=True)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Обычный подход: GridSearchCV с 5-фолдовой CV
param_grid = {'C': [0.1, 1, 10, 100], 'gamma': [0.001, 0.01, 0.1, 1]}
grid = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, scoring='accuracy')
grid.fit(X_scaled, y)
naive_score = grid.best_score_
print(f"Обычная CV (оптимистичная): {naive_score:.4f}")

# Nested CV: внешний цикл 5-fold, внутренний 5-fold GridSearchCV
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

nested_scores = []
for train_idx, test_idx in outer_cv.split(X_scaled, y):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    # Внутренний GridSearch
    inner_grid = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=inner_cv, scoring='accuracy')
    inner_grid.fit(X_train, y_train)
    best_model = inner_grid.best_estimator_
    # Оценка на внешнем тесте
    test_score = best_model.score(X_test, y_test)
    nested_scores.append(test_score)

nested_score = np.mean(nested_scores)
print(f"Nested CV (честная):     {nested_score:.4f}")
print(f"Разница (смещение):      {naive_score - nested_score:.4f}")
```

Типичный вывод покажет, что наивная оценка завышена на несколько процентных пунктов, а nested CV даёт более консервативную и реалистичную цифру.

**Вычислительная сложность.** При $K_{\text{out}}$ внешних фолдах и $K_{\text{in}}$ внутренних для сетки из $M$ точек требуется обучить модель примерно $M \times K_{\text{out}} \times K_{\text{in}}$ раз. Это существенно больше, чем простой GridSearchCV, но зато даёт честную оценку, что особенно важно при сравнении различных методов или при подготовке итоговых отчётов.

### 3. Bias‑variance разложение для ансамблей

Ансамблевые методы, такие как бэггинг (bagging) и бустинг (boosting), объединяют предсказания нескольких базовых моделей и часто значительно превосходят по качеству каждую из них. Разложение ошибки на смещение и дисперсию помогает понять, почему это происходит.

**Бэггинг** (bootstrap aggregating) строит $B$ независимых моделей на бутстреп‑выборках и усредняет их предсказания:
$$
\bar{f}(\mathbf{x}) = \frac{1}{B} \sum_{b=1}^{B} \hat{f}_b(\mathbf{x}).
$$
Каждая $\hat{f}_b$ обучена на своей псевдовыборке, полученной случайным вытягиванием с возвращением из исходных данных. Благодаря этому базовые модели становятся декоррелированными.

Пусть $\mathrm{Var}(\hat{f}_b) = \sigma^2$ и средняя корреляция между предсказаниями двух разных моделей равна $\rho$. Тогда дисперсия усреднённого предсказания составляет
$$
\mathrm{Var}(\bar{f}) = \rho \sigma^2 + \frac{1-\rho}{B} \sigma^2.
\tag{3.1}
$$
При $B \to \infty$ дисперсия стремится к $\rho \sigma^2$. Если модели не полностью коррелированы ($\rho < 1$), то $\mathrm{Var}(\bar{f}) < \sigma^2$. Бэггинг практически не изменяет смещение, но существенно снижает дисперсию, что особенно полезно для нестабильных моделей с высокой дисперсией (например, глубоких решающих деревьев). Случайный лес добавляет ещё и случайный отбор признаков, дополнительно уменьшая $\rho$ и, следовательно, дисперсию.

**Бустинг** (например, AdaBoost, градиентный бустинг) строит ансамбль последовательно: каждая следующая модель фокусируется на ошибках предыдущих. В терминах bias‑variance бустинг в первую очередь уменьшает смещение, последовательно усложняя решающее правило. Однако при чрезмерном числе итераций он может начать увеличивать дисперсию и переобучаться. На практике градиентный бустинг с регуляризацией (learning rate, ограничение глубины деревьев) обеспечивает превосходный баланс.

> **Углублённый взгляд: вывод дисперсии усреднённого предсказания**
>
> Пусть $\hat{f}_1, \dots, \hat{f}_B$ — случайные величины с одинаковой дисперсией $\sigma^2$ и корреляцией $\rho = \text{Corr}(\hat{f}_i, \hat{f}_j)$ при $i \neq j$. Дисперсия среднего равна
> $$
> \begin{aligned}
> \mathrm{Var}(\bar{f}) &= \mathrm{Var}\!\left(\frac{1}{B}\sum_{i=1}^B \hat{f}_i\right)
> = \frac{1}{B^2} \left( \sum_{i=1}^B \mathrm{Var}(\hat{f}_i) + \sum_{i \neq j} \mathrm{Cov}(\hat{f}_i, \hat{f}_j) \right) \\
> &= \frac{1}{B^2} \left( B\sigma^2 + B(B-1)\rho\sigma^2 \right)
> = \frac{\sigma^2}{B} + \frac{B-1}{B}\rho\sigma^2 \\
> &= \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2.
> \end{aligned}
> $$
> Если модели некоррелированы ($\rho = 0$), дисперсия уменьшается в $B$ раз. Если они идентичны ($\rho = 1$), усреднение не даёт выигрыша. Реальные бэггинг‑ансамбли находятся между этими крайностями, обеспечивая значительное снижение дисперсии.

### 4. Итоговое сравнение и рекомендации

Сведём рассмотренные методы валидации и подбора гиперпараметров в единую таблицу.

| Метод | Смещение оценки | Дисперсия оценки | Вычислительная сложность | Применимость к зависимым данным |
|-------|-----------------|------------------|---------------------------|----------------------------------|
| Однократный train/test split | Зависит от разбиения | Очень высокая | Низкая | Только при независимости |
| k‑fold CV | Низкое (k≥10) | Умеренная | $k$ обучений | Нет (без модификаций) |
| LOOCV | Очень низкое | Высокая | $n$ обучений | Нет |
| Стратифицированный k‑fold | Низкое | Умеренная | $k$ обучений | Нет |
| Repeated k‑fold | Низкое | Низкая | $k \times r$ обучений | Нет |
| TimeSeriesSplit | Смещение вверх (пессимистично) | Умеренная | $k$ обучений | Да |
| Group k‑fold | Низкое | Зависит от числа групп | $k$ обучений | Да, для групп |
| Nested CV | Практически несмещённое | Зависит от $K_{\text{out}}$ | $K_{\text{out}} \times K_{\text{in}} \times M$ обучений | Можно адаптировать |

**Практические рекомендации:**
- Для стандартных табличных данных с независимыми наблюдениями используйте **10‑fold стратифицированную кросс‑валидацию**, а при необходимости более точной оценки дисперсии — **repeated 5×2‑fold** или **10‑fold с повторениями**.
- Для **временных рядов** — только последовательные схемы (TimeSeriesSplit с фиксированным или растущим окном). Обычный k‑fold здесь недопустим.
- Для данных с **групповой структурой** (пациенты, пользователи) всегда применяйте **GroupKFold** или его стратифицированный вариант.
- При подборе гиперпараметров с ограниченным бюджетом начинайте со **случайного поиска**, который даёт хорошее покрытие при любой размерности. Если обучение модели дорого́, переходите к **байесовской оптимизации**.
- Для итоговой оценки качества всего пайплайна (включая отбор признаков, выбор гиперпараметров) используйте **nested CV**. Это особенно важно при сравнении нескольких алгоритмов.
- При работе с ансамблями помните: **бэггинг и случайный лес** снижают дисперсию и идеальны для нестабильных моделей; **градиентный бустинг** при правильной регуляризации обеспечивает лучший баланс смещения и дисперсии.

Эти принципы лежат в основе современных систем автоматического машинного обучения (AutoML), которые комбинируют байесовскую оптимизацию, вложенную валидацию и ансамблирование для поиска оптимальных решений без ручного вмешательства.

### 5. Заключение и резюме главы

В этой главе мы проследили путь от фундаментального разложения ошибки до конкретных инструментов выбора и оценки модели. Ключевые концепции таковы:

- **Ожидаемая ошибка** предсказания состоит из неустранимого шума, квадрата смещения и дисперсии. Уменьшение одного компонента часто влечёт рост другого — это и есть **bias‑variance tradeoff**.
- **Кросс‑валидация** позволяет оценить ожидаемую ошибку по конечной выборке, снижая дисперсию оценки по сравнению с однократным разбиением. Выбор конкретной схемы (k‑fold, LOOCV, стратифицированная, групповая, временна́я) диктуется структурой данных.
- **Выбор гиперпараметров** — задача оптимизации по дорогостоящей целевой функции. Grid search систематичен, но страдает от проклятия размерности; random search эффективнее в высоких размерностях; байесовская оптимизация максимально использует каждый эксперимент.
- Для получения честной оценки всего процесса необходимо отделять данные для выбора модели от данных для её оценки — эту роль выполняет **nested CV**.
- **Ансамбли** манипулируют смещением и дисперсией: бэггинг понижает дисперсию, бустинг — смещение, а вместе они позволяют достичь выдающегося качества на практике.

Итоговая таблица связывает смещение и дисперсию с характеристиками модели и данных:

| Фактор | Влияние на смещение | Влияние на дисперсию |
|--------|---------------------|----------------------|
| Увеличение сложности модели | Снижает | Повышает |
| Увеличение объёма выборки $n$ | Не влияет | Снижает |
| Регуляризация (рост $\lambda$) | Повышает | Снижает |
| Добавление признаков | Может снижать | Может повышать |
| Бэггинг | Почти не меняет | Снижает |
| Бустинг | Снижает | Может повышать |

Знание этих зависимостей позволяет осознанно проектировать процесс построения модели, диагностировать проблемы и выбирать правильные инструменты.

---

## Вопросы для самопроверки (объединяют всю главу)

### Для начинающих

1. Из каких трёх компонентов складывается ожидаемая ошибка предсказания для регрессии? Какая из них принципиально неустранима?
2. Чем k‑fold кросс‑валидация лучше однократного разбиения на обучение и тест?
3. Почему для временных рядов нельзя использовать обычную перемешанную кросс‑валидацию? Какую схему следует применять?
4. Что даёт вложенная (nested) кросс‑валидация по сравнению с обычной? В какой ситуации она необходима?
5. Какой метод подбора гиперпараметров вы выберете при 5 гиперпараметрах и времени обучения одной модели в 10 минут? Обоснуйте выбор.
6. Почему бэггинг снижает дисперсию модели? Приведите интуитивное объяснение.

### Для профессионалов

1. Выведите разложение дисперсии усреднённого предсказания в бэггинге (формула 3.1) и покажите, что при $\rho < 1$ и $B > 1$ дисперсия строго меньше дисперсии одного базового ученика.
2. Докажите, что оценка качества, полученная той же кросс‑валидацией, что использовалась для выбора модели, смещена вниз (оптимистична). Приведите математическое обоснование.
3. Сравните Expected Improvement и Probability of Improvement как функции приобретения в байесовской оптимизации. Какая из них более склонна к exploitation, а какая — к exploration?
4. Покажите, что для линейной регрессии ошибка LOOCV может быть вычислена по формуле PRESS, и объясните связь LOOCV с информационным критерием AIC.
5. Предложите способ построения доверительного интервала для оценки качества, полученной nested CV, не ограничиваясь точечной оценкой.

---

## Задания повышенной сложности

1. **Реализация LOOCV с нуля.** Для линейной регрессии напишите функцию, вычисляющую LOOCV‑ошибку по аналитической формуле через диагональные элементы проекционной матрицы (формула PRESS). Убедитесь, что результат совпадает с поточечным обучением.
2. **Эксперимент с bias‑variance кривыми.** Для полиномиальной регрессии степени от 1 до 15 сгенерируйте много обучающих выборок из одной и той же истинной модели. Постройте графики смещения, дисперсии и полной ошибки как функции степени полинома. Подтвердите U‑образную форму.
3. **Ручная байесовская оптимизация.** Реализуйте с нуля байесовскую оптимизацию для подбора $C$ и $\gamma$ SVM, используя гауссовский процесс и Expected Improvement. Сравните сходимость с `RandomizedSearchCV` по графику «лучшее найденное качество vs число итераций».
4. **Доказательство несмещённости nested CV.** Формально докажите, что при $K_{\text{out}} \to n$ (внешний LOOCV) nested CV даёт строго несмещённую оценку ожидаемой ошибки процедуры «внутренний выбор + обучение».
5. **Сравнение CV для временных рядов.** Для процесса AR(2) сгенерируйте ряд длиной 200. Сравните оценки ошибки прогноза, полученные обычным 10‑fold CV и TimeSeriesSplit, с истинной ошибкой на длинном будущем горизонте. Покажите, насколько оптимистичен обычный CV.